# Inspect Bronze Pageview Data

Use this notebook to inspect the local Bronze layer under `data/raw` without mutating files. It checks the manifest, compares it with local gzip files, validates sample gzip files, parses sample pageview rows with the package utilities, and builds quick summary tables.

The expensive checks are disabled by default. Turn them on only when you want a full verification pass.

## 1. Setup

In [ ]:
from __future__ import annotations

import gzip
import hashlib
import json
import sys
from collections import Counter
from itertools import islice
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from wikitrend.pageviews import parse_dump_filename, parse_pageview_line

RAW_DIR = ROOT / "data" / "raw" / "pageviews"
MANIFEST_PATH = ROOT / "data" / "raw" / "pageviews_manifest.json"

RAW_DIR, MANIFEST_PATH

## 2. Load Manifest

In [ ]:
assert RAW_DIR.exists(), f"Missing raw directory: {RAW_DIR}"
assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_df = pd.DataFrame(manifest.get("files", []))

if not manifest_df.empty:
    manifest_df["timestamp_hour"] = pd.to_datetime(manifest_df["timestamp_hour"], utc=True)
    manifest_df["local_path"] = manifest_df["relative_path"].map(lambda value: RAW_DIR / value)

summary = {
    "manifest_version": manifest.get("manifest_version"),
    "plan_id": manifest.get("plan_id"),
    "algorithm": manifest.get("algorithm"),
    "files": len(manifest_df),
    "total_size_gib": round(manifest_df["size_bytes"].sum() / 1024**3, 2) if not manifest_df.empty else 0,
    "first_hour": manifest_df["timestamp_hour"].min() if not manifest_df.empty else None,
    "last_hour": manifest_df["timestamp_hour"].max() if not manifest_df.empty else None,
}

summary

In [ ]:
manifest_df.head()

## 3. Compare Manifest With Local Files

In [ ]:
local_files = sorted(RAW_DIR.rglob("pageviews-*.gz"))
local_df = pd.DataFrame(
    {
        "filename": [path.name for path in local_files],
        "local_path": local_files,
        "actual_size_bytes": [path.stat().st_size for path in local_files],
    }
)

inventory_df = manifest_df.merge(
    local_df,
    on="filename",
    how="outer",
    suffixes=("_manifest", "_actual"),
    indicator=True,
)
inventory_df["size_matches"] = inventory_df["size_bytes"].eq(inventory_df["actual_size_bytes"])

inventory_summary = inventory_df["_merge"].value_counts().to_dict()
inventory_summary["size_mismatches"] = int((inventory_df["_merge"].eq("both") & ~inventory_df["size_matches"]).sum())
inventory_summary

In [ ]:
inventory_df.loc[
    inventory_df["_merge"].ne("both") | ~inventory_df["size_matches"],
    ["filename", "_merge", "size_bytes", "actual_size_bytes", "size_matches"],
]

## 4. Validate Gzip And Optional Hashes

The default check validates a small sample. Set `VALIDATE_ALL_GZIP = True` or `HASH_ALL_FILES = True` for a full pass.

In [ ]:
VALIDATE_ALL_GZIP = False
HASH_ALL_FILES = False
SAMPLE_FILES = 5


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_gzip_file(path: Path) -> None:
    with gzip.open(path, "rb") as handle:
        while handle.read(1024 * 1024):
            pass


check_df = manifest_df.copy()
if not VALIDATE_ALL_GZIP:
    check_df = check_df.head(SAMPLE_FILES)

results = []
for row in check_df.itertuples(index=False):
    path = RAW_DIR / row.relative_path
    result = {
        "filename": row.filename,
        "exists": path.exists(),
        "gzip_valid": None,
        "sha256_matches": None,
        "error": None,
    }
    try:
        validate_gzip_file(path)
        result["gzip_valid"] = True
        if HASH_ALL_FILES:
            result["sha256_matches"] = sha256_file(path) == row.sha256
    except Exception as exc:
        result["gzip_valid"] = False
        result["error"] = repr(exc)
    results.append(result)

validation_df = pd.DataFrame(results)
validation_df

## 5. Parse Sample Rows

In [ ]:
MAX_FILES_TO_SAMPLE = 6
LINES_PER_FILE = 20_000

records = []
parse_stats = []

for path in local_files[:MAX_FILES_TO_SAMPLE]:
    date_value, hour = parse_dump_filename(path.name)
    malformed = 0
    parsed = 0
    source_counter = Counter()

    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
        for line in islice(handle, LINES_PER_FILE):
            record = parse_pageview_line(line, date_value, hour)
            if record is None:
                malformed += 1
                continue
            parsed += 1
            source_counter[record.source_project] += 1
            records.append(record.to_dict())

    parse_stats.append(
        {
            "filename": path.name,
            "sampled_lines": parsed + malformed,
            "parsed_rows": parsed,
            "malformed_or_out_of_scope_rows": malformed,
            "seen_source_projects": dict(source_counter),
        }
    )

sample_df = pd.DataFrame(records)
parse_stats_df = pd.DataFrame(parse_stats)

parse_stats_df

In [ ]:
sample_df.head()

## 6. Sample-Level Summaries

In [ ]:
if sample_df.empty:
    print("No parsed records in sample.")
else:
    display(
        sample_df.groupby(["project", "access_mode"], dropna=False)
        .agg(rows=("page_title", "size"), views=("view_count", "sum"), bytes=("response_size", "sum"))
        .sort_values("views", ascending=False)
    )

In [ ]:
if not sample_df.empty:
    display(
        sample_df.sort_values("view_count", ascending=False)[
            ["date", "hour", "source_project", "normalized_title", "view_count", "response_size"]
        ].head(25)
    )

In [ ]:
if not sample_df.empty:
    hourly_sample = (
        sample_df.groupby(["date", "hour", "project"], dropna=False)["view_count"]
        .sum()
        .reset_index()
    )
    pivot = hourly_sample.pivot_table(
        index=["date", "hour"], columns="project", values="view_count", fill_value=0
    )
    ax = pivot.plot(figsize=(12, 5), title="Sampled views by project and hour")
    ax.set_xlabel("Sampled hour")
    ax.set_ylabel("Views in sampled rows")

## 7. Optional Full Line Counts

This can take time because it scans complete gzip files. Leave disabled for quick inspection.

In [ ]:
COUNT_ALL_LINES = False

if COUNT_ALL_LINES:
    line_counts = []
    for path in local_files:
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as handle:
            count = sum(1 for _ in handle)
        line_counts.append({"filename": path.name, "line_count": count})
    line_counts_df = pd.DataFrame(line_counts)
    display(line_counts_df.describe())
    display(line_counts_df.head())
else:
    print("Set COUNT_ALL_LINES = True to scan every Bronze gzip file.")

## 8. What To Look For

- Manifest file count should match the planned acquisition window.
- Inventory should show all files in both manifest and local storage.
- Size mismatches should be zero.
- Sample gzip validation should pass.
- Parsed rows should include the project/access combinations you expect.
- `malformed_or_out_of_scope_rows` includes unsupported Wikimedia project codes, not only broken lines.